# LB and SB Virtual screening pipeline (3D Align → 3D Dock → Boltz 3D Affinity)

This notebook chains the three **Iktos Engine** APIs used in 3D virtual screening:

1. **Iktos 3D Align** — ligand-based (LB) alignment / ranking on a reference active.
2. **Iktos 3D Dock** — structure-based (SB) docking into the protein pocket using the same reference.
3. **Boltz 3D Affinity** — affinity scoring on the **docked poses** returned by 3D Dock (no cofolding).

For API-specific details, see:

- `3d-align/3DAlign_api_example.ipynb`
- `3d-dock/3DDock_api_example.ipynb`
- `boltz-3d-affinity/Boltz_3Daffinity_api_example.ipynb`


## 1. Setup


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk upwards until we find the engine repository root."""
    markers = {"LICENSE", "README.md", "boltz-3d-affinity"}
    for p in (start, *start.parents):
        if all((p / m).exists() for m in markers):
            return p
    return start


# Notebook cwd is usually `virtual-screening-pipeline/`; repo root is one level up.
REPO_ROOT = find_repo_root(Path.cwd().resolve())

for rel in ("3d-align", "3d-dock", "boltz-3d-affinity"):
    d = REPO_ROOT / rel
    if d.is_dir():
        s = str(d)
        if s not in sys.path:
            sys.path.insert(0, s)

print("Repo root:", REPO_ROOT)
print("Added client folders to sys.path (if present).")


## 2. Setup — imports


In [ ]:
import json
import os

import pandas as pd

from align_3d_client import (
    Align3DClient,
    LigandInput as AlignLigandInput,
    ReferenceLigandInput,
)
from boltz_api_client import (
    AsyncApiClient,
    LigandInput as BoltzLigandInput,
    ProteinInput as BoltzProteinInput,
    load_molblock_file,
    load_pdb_file,
)
from dock_3d_client import (
    Dock3DClient,
    LigandInput as DockLigandInput,
    ProteinInput as DockProteinInput,
    ReferenceLigandInput as DockReferenceLigandInput,
)

## 3. API URLs, API key, and clients


In [ ]:
API_BASE_URL = os.getenv("API_BASE_URL")
API_KEY = os.getenv("API_KEY")

ALIGN_3D_API_URL = os.path.join(API_BASE_URL, "3d-align")
DOCK_3D_API_URL = os.path.join(API_BASE_URL, "3d-dock")
BOLTZ_AFFINITY_API_URL = os.path.join(API_BASE_URL, "boltz-3d-affinity")

align_client = Align3DClient(ALIGN_3D_API_URL, api_key=API_KEY)
dock_client = Dock3DClient(DOCK_3D_API_URL, api_key=API_KEY)
boltz_client = AsyncApiClient(BOLTZ_AFFINITY_API_URL, api_key=API_KEY)


## 4. Health checks & quota


In [ ]:
for name, client in (
    ("3D Align", align_client),
    ("3D Dock", dock_client),
    ("Boltz 3D Affinity", boltz_client),
):
    health = client.health_check()
    quota = client.get_quota()
    print(f"[{name}] API status: {health['status']}")
    print(f"[{name}] Quota: {quota.quota_used}/{quota.quota_max} used, {quota.quota_remaining} remaining")




## 5. Example data paths


In [ ]:
DATA_DIR = Path("./data/3kl6_example").resolve()

PROTEIN_RAW_PDB = DATA_DIR / "protein.pdb"
PROTEIN_CLEAN_DOCK_PDB = DATA_DIR / "protein_clean_3ddock.pdb"
PROTEIN_CLEAN_BOLTZ_PDB = DATA_DIR / "protein_clean_boltz.pdb"

REFERENCE_LIGAND_SDF = DATA_DIR / "ref_ligand.sdf"
SMILES_CSV = DATA_DIR / "smiles_list.csv"
MSA_CSV = DATA_DIR / "protein_3kl6_A.csv"

print("DATA_DIR:", DATA_DIR)


## 6. Clean protein PDB *(optional but recommended)*


In [ ]:
## Each api necessit a specific type of protein cleaning.
raw_pdb = load_pdb_file(str(PROTEIN_RAW_PDB))

clean_for_dock = dock_client.clean_protein(raw_pdb)
PROTEIN_CLEAN_DOCK_PDB.write_text(clean_for_dock, encoding="utf-8")
print("Wrote:", PROTEIN_CLEAN_DOCK_PDB)

clean_for_boltz = boltz_client.clean_protein(raw_pdb)
PROTEIN_CLEAN_BOLTZ_PDB.write_text(clean_for_boltz, encoding="utf-8")
print("Wrote:", PROTEIN_CLEAN_BOLTZ_PDB)


## 7. Prepare shared inputs (protein + reference ligand)


In [ ]:
dock_protein = DockProteinInput(
    id="protein_3kl6",
    mol_block=load_pdb_file(str(PROTEIN_CLEAN_DOCK_PDB)),
)

dock_reference = DockReferenceLigandInput(
    id="ligand_3kl6",
    mol_block=load_molblock_file(str(REFERENCE_LIGAND_SDF)),
)

align_reference = ReferenceLigandInput(
    id=dock_reference.id,
    mol_block=dock_reference.mol_block,
)


## 8. Stage A — 3D Align (LB screening)


In [ ]:
def show_progress_align_or_dock(request):
    s = request.status_summary
    if s is None:
        return
    print(
        f"  {s.completed}/{s.total} completed ({s.completed_percent:.1f}%)  |  failed: {s.failed}"
    )


ligands_data = json.loads(pd.read_csv(SMILES_CSV).to_json(orient="records"))
align_ligands = [AlignLigandInput(id=row["ligand_id"], smiles=row["smiles"]) for row in ligands_data]

align_response = align_client.create_request(
    reference_ligand=align_reference,
    ligands=align_ligands,
    batch_job_id="example_3kl6_lb_screening",
)

align_request_id = align_response["request_id"]
print(f"Request submitted. request_id: {align_request_id}")
print(f"Total ligands: {align_response['total_ligands']}")


In [ ]:
align_final = align_client.wait_for_completion(
    align_request_id,
    check_interval=5,
    progress_callback=show_progress_align_or_dock,
)
print(f"\nDone! Final status: {align_final.status.value}")


In [ ]:
align_results = align_client.get_results(align_request_id)

rows = []
for ligand_id, res in align_results.items():
    row = {"ligand_id": ligand_id, "status_3dalign": res["status"], "time_elapsed_3dalign": res.get("time_elapsed")}
    if res["status"] == "COMPLETED":
        data = res["data"]
        row.update(
            {
                "smiles": data.get("ligand_smiles"),
                "pharmaco_score": data.get("pharmaco_score"),
                "shape_score": data.get("shape_score"),
                "ligand_pose_sdf": data.get("ligand_pose_sdf"),
            }
        )
    elif res["status"] == "FAILED":
        row["error_3dalign"] = res.get("error")
    rows.append(row)

df_align = pd.DataFrame(rows)
cols = [c for c in df_align.columns if c != "ligand_pose_sdf"]
df_align[cols].head()


### 8.1 Filter aligned hits (example threshold)


In [ ]:
# Geometric mean of pharmaco + shape scores (tune for your project).
df_align["filtering_score"] = (df_align["pharmaco_score"] * df_align["shape_score"]) ** 0.5
df_align_stage2 = df_align[df_align["filtering_score"] > 0.3].copy()
len(df_align_stage2)


## 9. Stage B — 3D Dock (SB docking on filtered candidates)


In [ ]:
dock_ligands = [
    DockLigandInput(id=row["ligand_id"], smiles=row["smiles"])
    for _, row in df_align_stage2.iterrows()
]

dock_response = dock_client.create_request(
    protein=dock_protein,
    reference_ligand=dock_reference,
    ligands=dock_ligands,
    batch_job_id="example_3kl6_sb_screening",
)

dock_request_id = dock_response["request_id"]
print(f"Request submitted. request_id: {dock_request_id}")
print(f"Total ligands: {dock_response['total_ligands']}")


In [ ]:
dock_final = dock_client.wait_for_completion(
    dock_request_id,
    check_interval=5,
    progress_callback=show_progress_align_or_dock,
)
print(f"\nDone! Final status: {dock_final.status.value}")


In [ ]:
dock_results = dock_client.get_results(dock_request_id)

rows = []
for ligand_id, res in dock_results.items():
    row = {"ligand_id": ligand_id, "status_3ddock": res["status"], "time_elapsed_3ddock": res.get("time_elapsed")}
    if res["status"] == "COMPLETED":
        data = res["data"]
        row.update(
            {
                "ligand_smiles": data.get("ligand_smiles"),
                "vina_score": data.get("vina_score"),
                "pharmaco_score": data.get("pharmaco_score"),
                "shape_score": data.get("shape_score"),
                "ligand_pose_sdf": data.get("ligand_pose_sdf"),
            }
        )
    elif res["status"] == "FAILED":
        row["error_3ddock"] = res.get("error")
    rows.append(row)

df_dock = pd.DataFrame(rows)
df_dock.head()


### 9.1 Filter docked poses (example threshold)


In [ ]:
df_dock_for_boltz = df_dock[df_dock["vina_score"] < -8].copy()
len(df_dock_for_boltz)


## 10. Stage C — Boltz 3D Affinity (pose-level affinity)


In [ ]:
msa_filename = boltz_client.upload_msa(str(MSA_CSV))
print("MSA uploaded:", msa_filename)


In [ ]:
boltz_protein = BoltzProteinInput(
    name="protein_3kl6",
    pdb_block=load_pdb_file(str(PROTEIN_CLEAN_BOLTZ_PDB)),
    msa_filenames=[msa_filename],
)

boltz_ligands = []
for _, ligand in df_dock_for_boltz.iterrows():
    boltz_ligands.append(
        BoltzLigandInput(
            name=ligand["ligand_id"],
            molblock=ligand["ligand_pose_sdf"],
        )
    )


In [ ]:
boltz_response = boltz_client.create_request(
    job_name="sb_vs_3kl6_example",
    protein=boltz_protein,
    ligands=boltz_ligands,
)

boltz_request_id = boltz_response["parent_request_id"]
print(f"Request submitted. request_id: {boltz_request_id}")
print(f"Total ligands: {boltz_response['total_ligands']}")


In [ ]:
def show_progress_boltz(request):
    s = request.status_summary
    print(
        f"Pending: {s.pending}, Starting: {s.starting}, Processing: {s.processing}, "
        f"Completed: {s.completed}, Failed: {s.failed}"
    )
    print(f"  {s.completed}/{s.total} completed ({s.completed_percent:.1f}%)")


boltz_final = boltz_client.wait_for_completion(
    boltz_request_id,
    check_interval=10,
    progress_callback=show_progress_boltz,
)

print(f"\nDone! Final status: {boltz_final.status.value}")


In [ ]:
boltz_results = boltz_client.get_results(boltz_request_id)

rows = []
for ligand_name, result in boltz_results.items():
    row = {"ligand_id": ligand_name, "status_boltz3daffinity": result["status"]}
    if result["status"] == "COMPLETED":
        row.update(
            result["data"]
        )
    elif result["status"] == "FAILED":
        row["error_boltz3daffinity"] = result["error"]
    rows.append(row)

df_boltz = pd.DataFrame(rows)
df_boltz.head()


## 11. Save results *(optional)*


In [ ]:
df_out = df_boltz.merge(df_dock_for_boltz, on="ligand_id", how="inner")

out_csv = DATA_DIR / "virtual_screening_boltz_affinity_A.csv"
df_out.to_csv(out_csv, index=False)
print("Saved:", out_csv)